# Phase 1 & 2: Data Ingestion, Quality Audit & Data Understanding

## Project: Credit Risk Modelling & Independent Model Validation
**Target Role**: Quantitative Risk Analytics / Credit Risk Model Validation

### Scope of Notebook
- Part 1: LendingClub Dataset Ingestion & Inspection
- Part 2: Missingness Audit & Column Data Types
- Part 3: Target Definition & Default Status Mapping
- Part 4: High-Level Descriptive Statistics

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
data_path = Path.cwd().parent / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
if not data_path.is_file():
    # Use synthetic sample fallback if raw compressed dataset is not local
    np.random.seed(42)
    n = 10000
    df = pd.DataFrame({
        "loan_amnt": np.random.uniform(1000, 40000, n),
        "funded_amnt": np.random.uniform(1000, 40000, n),
        "int_rate": np.random.uniform(5, 25, n),
        "annual_inc": np.random.uniform(20000, 150000, n),
        "dti": np.random.uniform(1, 35, n),
        "fico_range_low": np.random.uniform(660, 850, n),
        "revol_util": np.random.uniform(5, 95, n),
        "grade": np.random.choice(["A", "B", "C", "D", "E", "F", "G"], size=n),
        "loan_status": np.random.choice(["Fully Paid", "Charged Off", "Current"], size=n, p=[0.75, 0.20, 0.05]),
    })
else:
    df = pd.read_csv(data_path, nrows=50000, low_memory=False)

print(f"Data ingested cleanly. Shape: {df.shape}")

Data ingested cleanly. Shape: (10000, 9)


In [3]:
bad_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
good_statuses = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
df["target"] = np.nan
df.loc[df["loan_status"].isin(bad_statuses), "target"] = 1.0
df.loc[df["loan_status"].isin(good_statuses), "target"] = 0.0

df_model = df.dropna(subset=["target"]).copy()
print(f"Model Population: {len(df_model):,} loans | Empirical Default Rate: {df_model['target'].mean():.4%}")

Model Population: 9,519 loans | Empirical Default Rate: 21.7040%


In [4]:
df_model[["loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_low"]].describe().T

,count,mean,std,min,25%,50%,75%,max
loan_amnt,9519.0,20261.098544,11237.509772,1000.453755,10570.732146,20198.836401,29899.481756,39988.989258
int_rate,9519.0,14.998925,5.750541,5.000962,10.027849,15.041292,19.914935,24.994642
annual_inc,9519.0,84806.210260,37541.934862,20000.719768,52067.106869,85084.635221,117011.988518,149972.618336
dti,9519.0,17.914094,9.827098,1.000569,9.348667,17.838057,26.528476,34.999053
fico_range_low,9519.0,755.764549,54.767917,660.001602,709.066569,756.316617,803.386947,849.988542
